In [ ]:
# Uninstall conflicting packages (if present)
import subprocess, sys
for pkg in ['keras', 'matplotlib', 'scikit-learn', 'tensorflow']:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '--yes', pkg], capture_output=True)

In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
import os
import sys
import subprocess

In [ ]:
import glob

def set_env(input_archive, temp_dir):
    archive = input_archive
    if not os.path.exists(archive):
        candidates = glob.glob('/kaggle/input/**/wheels.tar.gz', recursive=True)
        if candidates:
            archive = candidates[0]
            print(f'Found archive at: {archive}')
        else:
            print('No wheels archive found, using pre-installed packages')
            return

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        subprocess.run(['tar', '-xzf', archive, '-C', temp_dir], check=True)

    subprocess.run([
        sys.executable, '-m', 'pip', 'install',
        '--no-index', '--find-links', f'{temp_dir}/wheels',
        'unsloth', 'trl', 'vllm', 'openai_harmony'
    ], check=True)

    tk = os.path.join(temp_dir, 'tiktoken_encodings')
    if os.path.exists(tk):
        os.environ['TIKTOKEN_ENCODINGS_BASE'] = tk

In [ ]:
set_env(
    input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz',
    temp_dir='/kaggle/tmp/setup'
)

for tk in glob.glob('/kaggle/input/**/tiktoken_encodings', recursive=True) + glob.glob('/kaggle/tmp/**/tiktoken_encodings', recursive=True):
    os.environ['TIKTOKEN_ENCODINGS_BASE'] = tk
    print(f'TIKTOKEN: {tk}')
    break

model_path = '/kaggle/input/gpt-oss-120b/transformers/default/1'
if not os.path.exists(model_path):
    for candidate in glob.glob('/kaggle/input/**/config.json', recursive=True):
        d = os.path.dirname(candidate)
        if 'gpt-oss' in d.lower():
            model_path = d
            break
    print(f'Model path (fallback): {model_path}')
else:
    print(f'Model path: {model_path}')

In [ ]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [ ]:
import gc
import re
import math
import time
import glob
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName,
    load_harmony_encoding,
    SystemContent,
    ReasoningEffort,
    ToolNamespaceConfig,
    Author,
    Message,
    Role,
    TextContent,
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [ ]:
# =============================================================================
# COMPONENT A: MATHEMATICAL TOOL LIBRARY
# Pre-loaded into every Jupyter sandbox so the model can call these directly.
# Covers the key techniques needed for Problems 5-10 difficulty level.
# =============================================================================
MATH_TOOL_LIBRARY = '''
# ============================================================
# AIMO3 Mathematical Tool Library
# All functions available in this sandbox session.
# ============================================================
import math
import sympy
import numpy as np
import mpmath
import itertools
import collections
from sympy import (
    factorint, isprime, nextprime, totient, divisors, divisor_count,
    binomial, factorial, Rational, gcd, lcm, mod_inverse,
    Matrix, symbols, solve, simplify, expand, factor, Poly,
    floor as sym_floor, ceiling as sym_ceiling, sqrt as sym_sqrt,
    cos, sin, tan, pi, E, oo, zoo, nan, I
)
mpmath.mp.dps = 64

# ------------------------------------------------------------------
# Number Theory: p-adic valuation
# ------------------------------------------------------------------

def v_p(n, p):
    """p-adic valuation: largest k such that p^k | n.
    Works for arbitrary-size integers.
    v_p(0, p) = infinity (returned as float(\'inf\')).
    """
    if n == 0:
        return float(\'inf\')
    n = abs(int(n))
    count = 0
    while n % p == 0:
        count += 1
        n //= p
    return count


def legendre_v_p(n, p):
    """v_p(n!) via Legendre\'s formula: sum_{i>=1} floor(n/p^i).
    Exact for arbitrary n (no floating point).
    """
    n = int(n)
    total = 0
    pk = p
    while pk <= n:
        total += n // pk
        pk *= p
    return total


def v_p_binom(n, k, p):
    """v_p(C(n,k)) = v_p(n!) - v_p(k!) - v_p((n-k)!) via Kummer\'s theorem.
    Kummer: equals the number of carries when adding k and (n-k) in base p.
    """
    return legendre_v_p(n, p) - legendre_v_p(k, p) - legendre_v_p(n - k, p)


def v_p_factorial_exact(n, p):
    """Alias for legendre_v_p. Returns v_p(n!)."""
    return legendre_v_p(n, p)


# ------------------------------------------------------------------
# Lifting the Exponent Lemma (LTE)
# ------------------------------------------------------------------

def lte_odd(p, a, b, n):
    """LTE for odd prime p: if p | (a+b) and p does not divide a or b,
    then v_p(a^n + b^n) = v_p(a+b) + v_p(n)  [for odd n].
    If p | (a-b) and p does not divide a or b,
    then v_p(a^n - b^n) = v_p(a-b) + v_p(n).
    This function returns v_p(a^n - b^n) under the condition p | (a-b).
    """
    if v_p(a - b, p) == 0:
        raise ValueError(f\'LTE requires p | (a-b). Got v_{p}({a}-{b})=0.\')
    if v_p(a, p) > 0 or v_p(b, p) > 0:
        raise ValueError(f\'LTE requires p does not divide a or b.\')
    return v_p(a - b, p) + v_p(n, p)


def lte_sum_odd(p, a, b, n):
    """LTE for a^n + b^n with odd prime p and odd n: v_p(a^n+b^n) = v_p(a+b)+v_p(n).
    Requires: p | (a+b), p does not divide a or b, n is odd.
    """
    if n % 2 == 0:
        raise ValueError(\'LTE sum form requires odd n.\')
    if v_p(a + b, p) == 0:
        raise ValueError(f\'LTE requires p | (a+b).\')
    if v_p(a, p) > 0 or v_p(b, p) > 0:
        raise ValueError(f\'LTE requires p does not divide a or b.\')
    return v_p(a + b, p) + v_p(n, p)


def lte_p2(a, b, n):
    """LTE for p=2: v_2(a^n - b^n) = v_2(a-b) + v_2(a+b) + v_2(n) - 1.
    Requires: 2 | (a-b) i.e. a,b both odd, and n even. (a,b odd integers)
    """
    if (a - b) % 2 != 0:
        raise ValueError(\'LTE p=2 requires a-b even (both odd or both even).\')
    return v_p(a - b, 2) + v_p(a + b, 2) + v_p(n, 2) - 1


# ------------------------------------------------------------------
# Combinatorics: Catalan numbers and related
# ------------------------------------------------------------------

def catalan(n):
    """nth Catalan number C_n = C(2n,n)/(n+1). Exact integer via sympy."""
    return int(sympy.catalan(n))


def catalan_mod(n, m):
    """nth Catalan number mod m. Uses Lucas/direct computation."""
    return int(sympy.catalan(n)) % m


def ballot_number(n, k):
    """Ballot number: ways a candidate gets n votes and is always strictly ahead
    of another who gets k votes. = (n-k)/(n+k) * C(n+k, n).
    """
    if n <= k:
        return 0
    return int((n - k) * binomial(n + k, n) // (n + k))


def count_trailing_zeros_factorial(n, base=10):
    """Largest k such that base^k | n!. For base=10: min(v_2(n!), v_5(n!))."""
    factors = factorint(base)
    return min(legendre_v_p(n, p) // e for p, e in factors.items())


# ------------------------------------------------------------------
# Sequences: Fibonacci via matrix exponentiation (exact)
# ------------------------------------------------------------------

def fib_matrix(n):
    """Exact Fibonacci F(n) via matrix exponentiation. F(0)=0, F(1)=1.
    Works for arbitrarily large n in O(log n) matrix multiplications.
    """
    if n < 0:
        # F(-n) = (-1)^(n+1) * F(n)
        sign = (-1) ** (abs(n) + 1)
        return sign * fib_matrix(abs(n))
    if n == 0: return 0
    if n == 1: return 1

    def mat_mul(A, B):
        return [
            [A[0][0]*B[0][0] + A[0][1]*B[1][0], A[0][0]*B[0][1] + A[0][1]*B[1][1]],
            [A[1][0]*B[0][0] + A[1][1]*B[1][0], A[1][0]*B[0][1] + A[1][1]*B[1][1]]
        ]

    def mat_pow(M, p):
        if p == 1: return M
        if p % 2 == 0:
            half = mat_pow(M, p // 2)
            return mat_mul(half, half)
        return mat_mul(M, mat_pow(M, p - 1))

    M = [[1, 1], [1, 0]]
    result = mat_pow(M, n)
    return result[0][1]


def fib(n):
    """Alias for fib_matrix."""
    return fib_matrix(n)


def fib_mod(n, m):
    """F(n) mod m using matrix exponentiation with modular arithmetic.
    Much faster than fib(n) % m for large n due to Pisano period.
    """
    if n < 0:
        raise ValueError(\'fib_mod requires n >= 0.\')
    if n == 0: return 0
    if n == 1: return 1

    def mat_mul_mod(A, B, mod):
        return [
            [(A[0][0]*B[0][0] + A[0][1]*B[1][0]) % mod, (A[0][0]*B[0][1] + A[0][1]*B[1][1]) % mod],
            [(A[1][0]*B[0][0] + A[1][1]*B[1][0]) % mod, (A[1][0]*B[0][1] + A[1][1]*B[1][1]) % mod]
        ]

    def mat_pow_mod(M, p, mod):
        if p == 1: return M
        if p % 2 == 0:
            half = mat_pow_mod(M, p // 2, mod)
            return mat_mul_mod(half, half, mod)
        return mat_mul_mod(M, mat_pow_mod(M, p - 1, mod), mod)

    M = [[1, 1], [1, 0]]
    result = mat_pow_mod(M, n, m)
    return result[0][1]


def golden_ratio_power(n, precision=50):
    """phi^n where phi=(1+sqrt(5))/2, computed with mpmath for high precision."""
    with mpmath.workdps(precision):
        phi = (1 + mpmath.sqrt(5)) / 2
        return mpmath.power(phi, n)


# ------------------------------------------------------------------
# Floor sums and Hermite\'s Identity
# ------------------------------------------------------------------

def floor_sum(n, a, b, m):
    """Compute sum_{i=0}^{n-1} floor((a*i + b) / m) exactly.
    Uses the algorithm from Knuth / competitive programming.
    All inputs must be non-negative integers with m > 0.
    """
    n, a, b, m = int(n), int(a), int(b), int(m)
    ans = (a // m) * n * (n - 1) // 2 + (b // m) * n
    a %= m
    b %= m
    y_max = (a * n + b) // m
    x_max = y_max * m - b
    if y_max == 0:
        return ans
    ans += (n - (x_max + a - 1) // a) * y_max
    ans += floor_sum(y_max, m, (a - x_max % a) % a, a)
    return ans


def hermite_floor_identity(x, n):
    """Hermite\'s identity: floor(x) + floor(x + 1/n) + ... + floor(x + (n-1)/n) = floor(nx).
    Returns True if identity holds for given x (mpmath float) and positive integer n.
    """
    with mpmath.workdps(50):
        x_mp = mpmath.mpf(x)
        lhs = sum(int(mpmath.floor(x_mp + mpmath.mpf(k) / n)) for k in range(n))
        rhs = int(mpmath.floor(n * x_mp))
        return lhs == rhs, lhs, rhs


# ------------------------------------------------------------------
# Number Theory: multiplicative functions, divisor sums
# ------------------------------------------------------------------

def sigma(n, k=1):
    """Sum of k-th powers of divisors of n. sigma_k(n).
    sigma_0 = number of divisors, sigma_1 = sum of divisors.
    Uses sympy.divisor_sigma internally.
    """
    return int(sympy.divisor_sigma(n, k))


def euler_phi(n):
    """Euler\'s totient phi(n)."""
    return int(sympy.totient(n))


def mobius(n):
    """Mobius function mu(n)."""
    return int(sympy.mobius(n))


def is_multiplicative_check(f, limit=50):
    """Verify if a dict/function f is multiplicative: f(mn)=f(m)f(n) for gcd(m,n)=1.
    Tests all pairs m,n in [1..limit] with gcd=1.
    f can be a callable f(n) or a dict {n: value}.
    """
    from math import gcd
    if isinstance(f, dict):
        get = lambda n: f.get(n, None)
    else:
        get = f
    for m in range(1, limit + 1):
        for n in range(1, limit + 1):
            if gcd(m, n) == 1:
                fm, fn, fmn = get(m), get(n), get(m * n)
                if None in (fm, fn, fmn): continue
                if fm * fn != fmn:
                    return False, m, n
    return True, None, None


def primitive_roots(p):
    """Find all primitive roots modulo prime p."""
    from sympy.ntheory import primitive_root, n_order
    g = primitive_root(p)
    phi = p - 1
    return [pow(g, k, p) for k in range(1, phi + 1) if math.gcd(k, phi) == 1]


# ------------------------------------------------------------------
# Polynomial tools: divisibility, Laurent polynomials
# ------------------------------------------------------------------

def poly_divides(f, g, ring=None):
    """Check if polynomial f divides g (g % f == 0) over the integers.
    f, g: sympy Poly objects or expressions.
    Returns (True, quotient) or (False, remainder).
    """
    x = symbols(\'x\')
    if not isinstance(f, sympy.Poly):
        f = Poly(f, x)
    if not isinstance(g, sympy.Poly):
        g = Poly(g, x)
    q, r = sympy.div(g, f, x, domain=\'ZZ\')
    return (True, q) if r == 0 else (False, r)


def convolution_product(alpha_dict, beta_dict):
    """Compute convolution (alpha * beta)(n) = sum_k alpha(k) * beta(n-k).
    alpha_dict, beta_dict: {index: value} for finitely-supported functions Z->Z.
    Returns dict of nonzero values.
    """
    result = collections.defaultdict(int)
    for i, ai in alpha_dict.items():
        for j, bj in beta_dict.items():
            result[i + j] += ai * bj
    return {k: v for k, v in result.items() if v != 0}


def shift_function(alpha_dict, n):
    """Shift operator S_n: (S_n alpha)(t) = alpha(t+n). Returns shifted dict."""
    return {k - n: v for k, v in alpha_dict.items()}


def inner_product(alpha_dict, beta_dict):
    """Inner product alpha * beta = sum_n alpha(n) * beta(n).
    (This is the dot-product / pointwise inner product, not convolution.)
    """
    result = 0
    all_keys = set(alpha_dict) | set(beta_dict)
    for k in all_keys:
        result += alpha_dict.get(k, 0) * beta_dict.get(k, 0)
    return result


def shifty_check(alpha_dict, beta_dict, k, l):
    """Check if S_n(alpha) star beta = 1 for n in {k,l}, 0 otherwise.
    Uses the inner-product definition of star from the problem.
    S_n(alpha)(t) = alpha(t+n), so (S_n(alpha) star beta) = sum_t alpha(t+n)*beta(t)
    = sum_t alpha(t) * beta(t-n)  [substituting t -> t-n]
    This is the cross-correlation of alpha and beta at lag n.
    """
    def star_at_n(n):
        return inner_product(shift_function(alpha_dict, -n), beta_dict)

    results = {}
    # Check a range to find support
    for n in range(-20, 21):
        val = star_at_n(n)
        if val != 0:
            results[n] = val
    return results


# ------------------------------------------------------------------
# Geometry helpers
# ------------------------------------------------------------------

def stewart_cevian(a, b, c, m, n):
    """Stewart\'s Theorem: In triangle ABC, cevian AD where D divides BC
    with BD=m, DC=n. Side AB=c, AC=b, BC=a=m+n.
    Returns length of cevian d^2 = (b^2*m + c^2*n - m*n*a) / a.
    """
    a_total = m + n
    d_sq = Rational(b**2 * m + c**2 * n, a_total) - m * n
    return d_sq


def angle_bisector_length(a, b, c):
    """Length of angle bisector from A in triangle with sides a=BC, b=CA, c=AB.
    t_a^2 = bc * [1 - (a/(b+c))^2].
    Returns t_a^2 as a Rational.
    """
    return Rational(b * c * ((b + c)**2 - a**2), (b + c)**2)


def triangle_area_heron(a, b, c):
    """Area of triangle with sides a, b, c via Heron\'s formula. Returns sympy expression."""
    s = Rational(a + b + c, 2)
    return sympy.sqrt(s * (s - a) * (s - b) * (s - c))


def pythagorean_triples(limit, primitive_only=True):
    """Generate Pythagorean triples (a,b,c) with c <= limit.
    If primitive_only, returns primitive triples only.
    """
    triples = []
    for m in range(2, int(limit**0.5) + 1):
        for n in range(1, m):
            if (m - n) % 2 == 1 and math.gcd(m, n) == 1:
                a = m*m - n*n
                b = 2*m*n
                c = m*m + n*n
                if c > limit: break
                triple = tuple(sorted([a, b, c]))
                triples.append(triple)
    if not primitive_only:
        all_triples = []
        for (a, b, c) in triples:
            k = 1
            while k * c <= limit:
                all_triples.append((k*a, k*b, k*c))
                k += 1
        return sorted(set(all_triples))
    return sorted(triples)


# ------------------------------------------------------------------
# Functional equations helpers
# ------------------------------------------------------------------

def find_multiplicative_functions(max_n=100, f1=None, constraint=None):
    """Enumerate multiplicative functions f: Z_>=1 -> Z_>=1 on [1..max_n].
    f1: forced value of f(1) (default 1 by multiplicativity).
    constraint: callable(n, val) returning True if val is a valid f(n).
    Returns list of valid function dicts.
    NOTE: This is a scaffold. For specific problems, adapt directly.
    """
    # By multiplicativity, f is determined by values on prime powers.
    # f(p^k) can be any value satisfying the constraint.
    primes_list = list(sympy.primerange(2, max_n + 1))
    prime_powers = {}
    for p in primes_list:
        pows = []
        pk = p
        while pk <= max_n:
            pows.append(pk)
            pk *= p
        prime_powers[p] = pows
    return prime_powers  # Return structure for caller to use


def functional_eq_Z_check(f_dict, eq_lhs, eq_rhs, test_range=50):
    """Verify a functional equation f(lhs) = rhs for all values in test_range.
    f_dict: dict mapping integers to integers.
    eq_lhs, eq_rhs: callables (m, n) -> integer (the argument to f).
    """
    failures = []
    for m in range(-test_range, test_range + 1):
        for n in range(-test_range, test_range + 1):
            try:
                lhs_arg = eq_lhs(m, n)
                rhs_arg = eq_rhs(m, n)
                lhs_val = f_dict.get(lhs_arg)
                rhs_val = f_dict.get(rhs_arg)
                if lhs_val is not None and rhs_val is not None and lhs_val != rhs_val:
                    failures.append((m, n, lhs_arg, rhs_arg, lhs_val, rhs_val))
                    if len(failures) >= 5: return failures
            except: pass
    return failures


# ------------------------------------------------------------------
# Graph theory helpers (for blackboard/sequence problems)
# ------------------------------------------------------------------

def digit_sum(n, base=10):
    """Sum of digits of n in given base."""
    n = abs(int(n))
    total = 0
    while n > 0:
        total += n % base
        n //= base
    return total


def iterated_digit_sum(n, base=10, steps=None):
    """Iteratively apply digit_sum until 1-digit (or steps exhausted).
    Returns (final_value, steps_taken).
    """
    count = 0
    while n >= base and (steps is None or count < steps):
        n = digit_sum(n, base)
        count += 1
    return n, count


def max_moves_to_1(n):
    """For the blackboard problem: given n, find the max moves (digit sum reductions)
    to reach 1, optimizing over base choice at each step.
    Memoized BFS/DP for moderate n.
    """
    from functools import lru_cache

    @lru_cache(maxsize=None)
    def best(m):
        if m == 1: return 0
        best_moves = 0
        for base in range(2, m + 1):
            s = digit_sum(m, base)
            if s < m:  # always true for base >= 2 and m >= 2
                moves = 1 + best(s)
                if moves > best_moves:
                    best_moves = moves
        return best_moves

    return best(int(n))


# ------------------------------------------------------------------
# Modular arithmetic helpers
# ------------------------------------------------------------------

def crt(remainders, moduli):
    """Chinese Remainder Theorem. Returns x such that x ≡ r_i (mod m_i) for all i.
    Moduli must be pairwise coprime.
    """
    from sympy.ntheory.modular import crt as sympy_crt
    return sympy_crt(moduli, remainders)


def discrete_log(a, b, m):
    """Baby-step giant-step: find x such that a^x ≡ b (mod m).
    Returns smallest non-negative x, or None if no solution.
    """
    return sympy.discrete_log(m, b, a)


def order_mod(a, m):
    """Multiplicative order of a modulo m: smallest k>0 with a^k ≡ 1 (mod m)."""
    return sympy.n_order(a, m)


# ------------------------------------------------------------------
# Tournament / combinatorics helpers
# ------------------------------------------------------------------

def count_10_trailing_zeros(n):
    """Trailing zeros in n = min(v_2(n), v_5(n))."""
    return min(v_p(n, 2), v_p(n, 5))


def stirling_second(n, k):
    """Stirling numbers of the second kind S(n,k)."""
    return int(sympy.functions.combinatorial.numbers.stirling(n, k))


def bell_number(n):
    """Bell number B_n."""
    return int(sympy.bell(n))


# Print confirmation
print("[MATH TOOLS LOADED] Available: v_p, legendre_v_p, v_p_binom, lte_odd, lte_sum_odd, lte_p2,")
print("  catalan, ballot_number, count_trailing_zeros_factorial,")
print("  fib, fib_matrix, fib_mod, golden_ratio_power,")
print("  floor_sum, hermite_floor_identity,")
print("  sigma, euler_phi, mobius, primitive_roots,")
print("  poly_divides, convolution_product, shift_function, inner_product, shifty_check,")
print("  stewart_cevian, angle_bisector_length, triangle_area_heron, pythagorean_triples,")
print("  digit_sum, iterated_digit_sum, max_moves_to_1,")
print("  crt, discrete_log, order_mod, stirling_second, bell_number")
'''

print('Math tool library defined:', len(MATH_TOOL_LIBRARY), 'chars')

In [ ]:
# =============================================================================
# COMPONENT B: PROBLEM TYPE DETECTOR
# Fast keyword-based classifier that runs in <1ms on CPU before inference.
# Returns domain label + confidence for selecting preference prompt.
# All regex patterns use Python 3.13-safe syntax (no bad escapes).
# =============================================================================

_DOMAIN_SIGNALS = {
    'number_theory': [
        r'\b(v_p|p-adic|valuation|divides|divisib|modulo|congruent)',
        r'\b(prime|primorial|factori[sz]|coprime|gcd|lcm|euler|totient)',
        r'\b(remainder|residue|digit.sum)',
        r'\b(multiplicative|additive|arithmetic function|divisor)',
        r'\b(legendre|lifting.*exponent|Kummer)',
        r'2\^k\b|p\^k\b|p-adic',
    ],
    'combinatorics': [
        r'\b(orderings?|arrangement|permutation|combination)',
        r'\b(Catalan|ballot|tournament|bracket|round)\b',
        r'\b(how many|number of.*possible|number of.*ways)',
        r'\b(path|lattice|tiling|coloring|subset)\b',
        r'\b(Stirling|Bell number|partition)\b',
        r'2\^',
        r'\b(runner|competitors?|ranking)\b',
    ],
    'geometry': [
        r'\b(triangle|circle|circumcircle|incircle|angle|bisect|tangent|chord)\b',
        r'\b(perpendicular|midpoint|collinear|concurrent|radical axis)\b',
        r'\b(Stewart|Ptolemy|Menelaus|Ceva|Heron)\b',
        r'\b(spiral similarity|cross.ratio|inversion|homothety)\b',
        r'\b(segment|perimeter|area|side length|altitude|median|cevian)\b',
        r'\b(Fibonacci.*triangle|triangle.*Fibonacci)\b',
        r'\b(cyclic|circumradius|inradius)\b',
    ],
    'functional_equations': [
        r'\b(function.*satisf|satisf.*equation|functional equation)\b',
        r'f\([mn]',
        r'f\(m.*n\)',
        r'[Zz].*->.*[Zz]|mathbb.*Z.*to',
        r'\b(for all.*positive integer|for all.*m.*n)\b',
        r'f\(m\) \+ f\(n\)',
    ],
    'sequences_recurrences': [
        r'\b(sequence|recurrence|Fibonacci|Lucas|recursive)\b',
        r'F_[0-9n]|a_[0-9n]|\(F_n\)',
        r'\b(closed form|general term|characteristic equation)\b',
        r'\b(Binet|golden ratio|varphi)\b',
    ],
    'floor_ceiling': [
        r'lfloor|lceil',
        r'\b(floor.*sum|sum.*floor|double sum)\b',
        r'\b(greatest integer|integer part|fractional part)\b',
        r'floor\(',
    ],
    'polynomial_algebra': [
        r'\b(polynomial|Laurent|formal power series|ring|ideal)\b',
        r'\b(degree|coefficient|irreducible)\b',
        r'\b(shifty|shift operator|convolution)\b',
        r'star|alpha.*beta|finitely.*supported',
        r'alpha.*in.*mathcal|mathcal.*F',
    ],
    'linear_algebra_systems': [
        r'\b(system of equations|simultaneous|linear equations)\b',
        r'\b(product.*age|age.*product|sum.*product)\b',
        r'\b(integer.*solution|solve.*integer)\b',
    ],
}


def detect_domain(problem_text):
    """Classify problem into mathematical domain(s).
    Returns list of (domain, score) sorted by score descending.
    Score = number of pattern matches (with try/except for regex safety).
    """
    text_lower = problem_text.lower()
    scores = {}
    for domain, patterns in _DOMAIN_SIGNALS.items():
        score = 0
        for pat in patterns:
            try:
                matches = re.findall(pat, text_lower, re.IGNORECASE)
                score += len(matches)
            except re.error:
                pass
        scores[domain] = score

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ranked


def primary_domain(problem_text, threshold=1):
    """Return the primary domain string, or 'general' if no strong signal."""
    ranked = detect_domain(problem_text)
    if ranked[0][1] >= threshold:
        return ranked[0][0]
    return 'general'


print('Domain detector defined.')


In [ ]:
# =============================================================================
# COMPONENT C: DOMAIN-SPECIFIC PREFERENCE PROMPTS
# Each prompt explicitly names the key theorems and Python tools.
# The model will see these as part of the user message for 2 of its 8 attempts.
# =============================================================================

PREF_NUMBER_THEORY = (
    'KEY TOOLS for number theory: '
    'v_p(n,p) gives p-adic valuation; legendre_v_p(n,p) gives v_p(n!) exactly. '
    'LTE: if odd prime p|(a-b), p not dividing a,b, then v_p(a^n-b^n)=v_p(a-b)+v_p(n). '
    'lte_odd(p,a,b,n) and lte_p2(a,b,n) implement these. '
    'For multiplicative f: determine f(p^k) on prime powers, then extend. '
    'sigma(n,k), euler_phi(n), factorint(n), mobius(n) all available. '
    'Use legendre_v_p for Legendre formula. Prefer sympy for exact arithmetic.'
)

PREF_COMBINATORICS = (
    'KEY TOOLS for combinatorics: '
    'catalan(n) gives exact Catalan number C_n=C(2n,n)/(n+1). '
    'binomial(n,k) for exact C(n,k). v_p_binom(n,k,p) for Kummer valuation. '
    'For tournament/ranking: think binary trees, score sequences, product formulas. '
    'count_trailing_zeros_factorial(n) uses Legendre for trailing zeros of n!. '
    'For large exponents in answers: apply Legendre formula directly. '
    'stirling_second(n,k), bell_number(n) also available. Verify with small cases.'
)

PREF_GEOMETRY = (
    'KEY TOOLS for geometry: '
    'stewart_cevian(a,b,c,m,n) gives cevian length^2 via Stewart theorem. '
    'angle_bisector_length(a,b,c) gives bisector from A. '
    'pythagorean_triples(limit) enumerates integer triangles. '
    'triangle_area_heron(a,b,c) for area. '
    'For cyclic quadrilaterals: use Ptolemy. For radical axis: power of a point. '
    'For Fibonacci triangle problems: fib(n) exact, then apply geometric constraints. '
    'Set up coordinates for angle chasing; use sympy.solve for systems.'
)

PREF_FUNCTIONAL_EQ = (
    'KEY APPROACH for functional equations: '
    'Substitute n=0, m=0, m=n, n=1, m=1 to discover structure. '
    'Key algebraic trick: m+n+mn = (m+1)(n+1)-1, so f maps shifted multiplicative structure. '
    'Check if f is determined by values at primes (multiplicativity). '
    'Use functional_eq_Z_check(f_dict, lhs, rhs, range) to verify candidates. '
    'Count valid f by counting valid assignments on generators/primes. '
    'Use sympy.solve for systems. Test f(n)=constant and f(n)=linear candidates.'
)

PREF_SEQUENCES = (
    'KEY TOOLS for sequences/recurrences: '
    'fib(n) exact Fibonacci via matrix exponentiation. fib_mod(n,m) for modular Fibonacci. '
    'Binet: F(n)=(phi^n - psi^n)/sqrt(5), phi=(1+sqrt(5))/2. '
    'golden_ratio_power(n) for high-precision phi^n. '
    'For limit of a_{2n} as n->inf: substitute Fibonacci ratios and compute. '
    'mpmath.mpf for 64-digit precision. Solve characteristic eqns with sympy.'
)

PREF_FLOOR = (
    'KEY TOOLS for floor/ceiling sums: '
    'floor_sum(n,a,b,m) computes sum_{i=0}^{n-1} floor((ai+b)/m) exactly. '
    'Hermite identity: floor(x)+floor(x+1/n)+...+floor(x+(n-1)/n)=floor(nx). '
    'hermite_floor_identity(x,n) verifies this. '
    'For double sums: swap summation order, express as sigma function. '
    'legendre_v_p(n,p) for exact v_p(n!). Use mpmath.floor for big floats. '
    'To find which 2^k | N: compute legendre_v_p carefully with divisibility.'
)

PREF_POLYNOMIAL = (
    'KEY TOOLS for polynomial/Laurent/shifty problems: '
    'convolution_product(alpha,beta) for convolution of {index:value} dicts. '
    'inner_product(alpha,beta) for star product: sum_n alpha(n)*beta(n). '
    'shift_function(alpha,n) implements shift operator S_n(alpha)(t)=alpha(t+n). '
    'shifty_check(alpha,beta,k,l) verifies the shifty condition. '
    'poly_divides(f,g) checks polynomial divisibility over ZZ. '
    'Note: (S_n(alpha) star beta) = cross-correlation of alpha and beta at lag n. '
    'Model finitely-supported Z->Z functions as Python dicts {int: int}.'
)

PREF_LINEAR_SYSTEMS = (
    'KEY APPROACH for word problems with systems of equations: '
    'Translate all conditions into algebraic equations. '
    'Use sympy.symbols and sympy.solve for exact integer solutions. '
    'Check all integer constraint solutions. '
    'Use sympy.Rational to avoid floating point. '
    'Consider parity and sign constraints to narrow solution space. '
    'Verify by substituting back into all original conditions.'
)

PREF_GENERAL = (
    'You have access to math tools: v_p, legendre_v_p, lte_odd, lte_p2, '
    'catalan, fib, fib_mod, floor_sum, sigma, euler_phi, '
    'poly_divides, convolution_product, shift_function, inner_product, '
    'stewart_cevian, angle_bisector_length, pythagorean_triples, '
    'digit_sum, crt, order_mod, stirling_second, bell_number.\n'
    'Also: math, numpy, sympy, mpmath (64-digit precision). '
    'Use sympy for exact symbolic answers. '
    'Always verify with small cases before generalizing. '
    'Prefer exact integer arithmetic over floating point.'
)

DOMAIN_TO_PREF = {
    'number_theory': PREF_NUMBER_THEORY,
    'combinatorics': PREF_COMBINATORICS,
    'geometry': PREF_GEOMETRY,
    'functional_equations': PREF_FUNCTIONAL_EQ,
    'sequences_recurrences': PREF_SEQUENCES,
    'floor_ceiling': PREF_FLOOR,
    'polynomial_algebra': PREF_POLYNOMIAL,
    'linear_algebra_systems': PREF_LINEAR_SYSTEMS,
    'general': PREF_GENERAL,
}

print('Domain-specific preference prompts defined for:', list(DOMAIN_TO_PREF.keys()))


In [ ]:
# =============================================================================
# CFG and all other classes (identical to v27, no vLLM param changes)
# =============================================================================
CFG_model_path = model_path

class CFG:
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'
        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'
        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'
        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.'
    )
    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Visualizing problem structure when helpful\n'
        '- Brute-force verification for small cases\n\n'
        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'
        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you\'re computing and why before running code.'
    )
    # Generic preference (used for 4 standard attempts)
    preference_prompt = PREF_GENERAL

    served_model_name = 'gpt-oss'
    model_path = CFG_model_path
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    # EXACT 44/50 vLLM params -- UNCHANGED
    high_problem_timeout = 900
    base_problem_timeout = 300
    notebook_limit = 17400
    server_timeout = 180
    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3
    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42
    gpu_memory_utilization = 0.96
    temperature = 0.8
    min_p = 0.02

print('CFG: v28 MathTools | same vLLM params as v27 + domain-aware prefs + tool library')

In [ ]:
set_seed(CFG.seed)

In [ ]:
class AIMO3Template:
    def __init__(self): pass

    def get_system_content(self, system_prompt, tool_config):
        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(self, system_prompt, user_prompt, tool_config):
        system_content = self.get_system_content(system_prompt, tool_config)
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)
        user_message = Message.from_role_and_content(Role.USER, user_prompt)
        return [system_message, user_message]

In [ ]:
class AIMO3Sandbox:
    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count=5):
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports

    def __init__(self, timeout, extra_init=''):
        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        self._extra_init = extra_init
        ports = self._get_next_ports(5)
        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'
        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]
        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])
        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True
        # Load standard libraries + math tool library
        base_init = 'import math\nimport numpy\nimport sympy\nimport mpmath\nimport itertools\nimport collections\nmpmath.mp.dps = 64\n'
        self.execute(base_init)
        if self._extra_init:
            self.execute(self._extra_init)

    def _format_error(self, traceback):
        clean = []
        for frame in traceback:
            cf = re.sub(r'\x1b\[[0-9;]*m', '', frame)
            if 'File "' in cf and 'ipython-input' not in cf: continue
            clean.append(cf)
        return ''.join(clean)

    def execute(self, code, timeout=None):
        client = self._client
        eff_timeout = timeout or self._default_timeout
        msg_id = client.execute(code, store_history=True, allow_stdin=False, stop_on_error=False)
        stdout_parts, stderr_parts = [], []
        start = time.time()
        while True:
            if time.time() - start > eff_timeout:
                self._km.interrupt_kernel()
                return f'[ERROR] Execution timed out after {eff_timeout} seconds'
            try: msg = client.get_iopub_msg(timeout=1.0)
            except queue.Empty: continue
            if msg.get('parent_header', {}).get('msg_id') != msg_id: continue
            mt = msg.get('msg_type')
            ct = msg.get('content', {})
            if mt == 'stream':
                (stdout_parts if ct.get('name') == 'stdout' else stderr_parts).append(ct.get('text', ''))
            elif mt == 'error': stderr_parts.append(self._format_error(ct.get('traceback', [])))
            elif mt in {'execute_result', 'display_data'}:
                t = ct.get('data', {}).get('text/plain')
                if t: stdout_parts.append(t if t.endswith('\n') else f'{t}\n')
            elif mt == 'status' and ct.get('execution_state') == 'idle': break
        so = ''.join(stdout_parts)
        se = ''.join(stderr_parts)
        if se: return f'{so.rstrip()}\n{se}' if so else se
        return so if so.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):
        with contextlib.suppress(Exception):
            if self._client: self._client.stop_channels()
        if self._owns_kernel and self._km:
            with contextlib.suppress(Exception): self._km.shutdown_kernel(now=True)
            with contextlib.suppress(Exception): self._km.cleanup_resources()

    def reset(self):
        base_init = 'import math\nimport numpy\nimport sympy\nimport mpmath\nimport itertools\nimport collections\nmpmath.mp.dps = 64\n'
        self.execute('%reset -f\n' + base_init)
        if self._extra_init:
            self.execute(self._extra_init)

    def __del__(self): self.close()

In [ ]:
class AIMO3Tool:
    def __init__(self, local_jupyter_timeout, tool_prompt, sandbox=None):
        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        self._owns_session = sandbox is None
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):
        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(
                        timeout=self._local_jupyter_timeout,
                        extra_init=MATH_TOOL_LIBRARY
                    )

    def _ensure_last_print(self, code):
        lines = code.strip().split('\n')
        if not lines: return code
        last = lines[-1].strip()
        if not last or 'print' in last or 'import' in last or last.startswith('#'): return code
        lines[-1] = 'print(' + last + ')'
        return '\n'.join(lines)

    @property
    def instruction(self): return self._tool_prompt

    @property
    def tool_config(self): return ToolNamespaceConfig(name='python', description=self.instruction, tools=[])

    def _make_response(self, output, channel=None):
        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')
        if channel: message = message.with_channel(channel)
        return message

    def process_sync_plus(self, message):
        self._ensure_session()
        raw = message.content[0].text
        final = self._ensure_last_print(raw)
        with self._execution_lock:
            try: output = self._jupyter_session.execute(final)
            except TimeoutError as exc: output = f'[ERROR] {exc}'
        return [self._make_response(output, channel=message.channel)]

In [ ]:
# Strategy-diverse preference prompts (same as v27)
PREF_CODE_FIRST = (
    'Solve this by writing a complete Python program. Go directly to code. '
    'Available: math, numpy, sympy, and all AIMO3 math tools (v_p, legendre_v_p, catalan, fib, etc). '
    'Your program must: 1) Compute the answer, '
    '2) Verify constraints, 3) Print the final answer. '
    'Prefer exact computation with sympy over floating point.'
)
PREF_SMALL_CASES = (
    'Start by testing small cases to find a pattern. '
    'If the problem involves n, try n=1,2,3,...,10. '
    'Write Python code to: 1) Compute results for small cases, '
    '2) Identify the pattern, 3) Verify for larger cases, '
    '4) Compute the final answer. '
    'Available: math, numpy, sympy, and all AIMO3 math tools (v_p, catalan, fib, floor_sum, etc).'
)

FOLLOWUP_PROMPT = (
    'You have been working on this problem. Based on your analysis so far, '
    'what is the final integer answer? The answer must be between 0 and 99999. '
    'Please state your answer inside \\boxed{}.'
)

print('Preference prompts defined.')

In [ ]:
# =============================================================================
# COMPONENT D: INTEGRATED SOLVER
# Key changes from v27:
#   1. AIMO3Sandbox now accepts extra_init= and loads MATH_TOOL_LIBRARY
#   2. solve_problem() detects domain and uses 2 domain-specialized attempts
#      in place of 2 of the 4 standard attempts
#   3. Attempt layout: [standard]*2 + [domain_a]*1 + [domain_b_if_2domains]*1
#        + [code_first]*2 + [small_cases]*2
#   4. The MATH_TOOL_LIBRARY is pre-loaded into every sandbox at init time,
#      so the model can call v_p(), catalan(), fib(), etc. immediately.
# =============================================================================

class AIMO3Solver:
    def __init__(self, cfg, port=8000):
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
        self._preload_model_weights()
        self.server_process = self._start_server()
        self.client = OpenAI(base_url=self.base_url, api_key=self.api_key, timeout=self.cfg.session_timeout)
        self._wait_for_server()
        self._initialize_kernels()
        self.notebook_start_time = time.time()
        self.problems_remaining = 50

    def _preload_model_weights(self):
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        t0 = time.time()
        files, total = [], 0
        for root, _, fnames in os.walk(self.cfg.model_path):
            for fn in fnames:
                fp = os.path.join(root, fn)
                if os.path.isfile(fp): files.append(fp); total += os.path.getsize(fp)
        def _read(p):
            with open(p, 'rb') as f:
                while f.read(1024*1024*1024): pass
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as ex: list(ex.map(_read, files))
        print(f'Processed {len(files)} files ({total/1e9:.2f} GB) in {time.time()-t0:.2f} seconds.\n')

    def _start_server(self):
        cmd = [
            sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
            '--seed', str(self.cfg.seed),
            '--model', self.cfg.model_path,
            '--served-model-name', self.cfg.served_model_name,
            '--tensor-parallel-size', '1',
            '--max-num-seqs', str(self.cfg.batch_size),
            '--gpu-memory-utilization', str(self.cfg.gpu_memory_utilization),
            '--host', '0.0.0.0', '--port', str(self.port),
            '--dtype', self.cfg.dtype,
            '--kv-cache-dtype', self.cfg.kv_cache_dtype,
            '--max-model-len', str(self.cfg.context_tokens),
            '--stream-interval', str(self.cfg.stream_interval),
            '--async-scheduling', '--disable-log-stats', '--enable-prefix-caching'
        ]
        self.log_file = open('vllm_server.log', 'w')
        return subprocess.Popen(cmd, stdout=self.log_file, stderr=subprocess.STDOUT, start_new_session=True)

    def _wait_for_server(self):
        print('Waiting for vLLM server...')
        t0 = time.time()
        for _ in range(self.cfg.server_timeout):
            rc = self.server_process.poll()
            if rc is not None:
                self.log_file.flush()
                with open('vllm_server.log') as f: logs = f.read()
                raise RuntimeError(f'Server died with code {rc}. Last 3000 chars:\n{logs[-3000:]}')
            try:
                self.client.models.list()
                print(f'Server is ready (took {time.time()-t0:.2f} seconds).\n')
                return
            except: time.sleep(1)
        raise RuntimeError('Server failed to start (timeout).')

    def _initialize_kernels(self):
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels with math tool library...')
        t0 = time.time()
        self.sandbox_pool = queue.Queue()
        # KEY CHANGE: pass MATH_TOOL_LIBRARY as extra_init to every sandbox
        def _mk(): return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout, extra_init=MATH_TOOL_LIBRARY)
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as ex:
            futs = [ex.submit(_mk) for _ in range(self.cfg.workers)]
            for f in as_completed(futs): self.sandbox_pool.put(f.result())
        print(f'Kernels initialized in {time.time()-t0:.2f} seconds.\n')

    def _scan_for_answer(self, text):
        for pat in [r'\\boxed\s*\{\s*([0-9,]+)\s*\}']:
            ms = re.findall(pat, text)
            if ms:
                try:
                    v = int(ms[-1].replace(',',''))
                    if 0 <= v <= 99999: return v
                except: pass
        ms = re.findall(r'\\boxed\s*\{\s*(-[0-9,]+)\s*\}', text)
        if ms:
            try:
                v = int(ms[-1].replace(',','')) % 100000
                if 0 <= v <= 99999: return v
            except: pass
        ms = re.findall(r'final\s+answer\s+is\s*([0-9,]+)', text, re.IGNORECASE)
        if ms:
            try:
                v = int(ms[-1].replace(',',''))
                if 0 <= v <= 99999: return v
            except: pass
        return None

    def _compute_mean_entropy(self, logprobs_buffer):
        if not logprobs_buffer: return float('inf')
        total, cnt = 0.0, 0
        for d in logprobs_buffer:
            if not isinstance(d, dict) or not d: continue
            e = 0.0
            for _, lp in d.items():
                p = math.exp(lp)
                if p > 0: e -= p * math.log2(p)
            total += e; cnt += 1
        return total / cnt if cnt else float('inf')

    def _process_attempt(self, problem, system_prompt, attempt_index, stop_event, deadline):
        if stop_event.is_set() or time.time() > deadline:
            return {'Attempt': attempt_index+1, 'Answer': None, 'Python Calls': 0, 'Python Errors': 0, 'Response Length': 0, 'Entropy': float('inf')}
        sandbox = None
        python_calls = python_errors = total_tokens = 0
        final_answer = None
        logprobs_buffer = []
        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
            local_tool = AIMO3Tool(local_jupyter_timeout=self.cfg.jupyter_timeout, tool_prompt=self.cfg.tool_prompt, sandbox=sandbox)
            encoding = self.encoding
            messages = self.template.apply_chat_template(system_prompt, problem, local_tool.tool_config)
            conversation = Conversation.from_messages(messages)
            for turn_idx in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline: break
                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
                if max_tokens < self.cfg.buffer_tokens: break
                stream = self.client.completions.create(
                    model=self.cfg.served_model_name, temperature=self.cfg.temperature,
                    logprobs=self.cfg.top_logprobs, max_tokens=max_tokens,
                    prompt=prompt_ids, seed=attempt_seed, stream=True,
                    extra_body={'min_p': self.cfg.min_p, 'stop_token_ids': self.stop_token_ids, 'return_token_ids': True}
                )
                try:
                    token_buffer, text_chunks = [], []
                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline: break
                        nt = chunk.choices[0].token_ids
                        nx = chunk.choices[0].text
                        if nt:
                            token_buffer.extend(nt); total_tokens += len(nt); text_chunks.append(nx)
                            clp = chunk.choices[0].logprobs
                            if clp and clp.top_logprobs: logprobs_buffer.extend(clp.top_logprobs)
                        if '}' in (nx or ''):
                            st = ''.join(text_chunks[-self.cfg.search_tokens:])
                            a = self._scan_for_answer(st)
                            if a is not None: final_answer = a; break
                finally: stream.close()
                if final_answer is not None: break
                if not token_buffer: break
                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last = new_messages[-1]
                if last.channel == 'final':
                    final_answer = self._scan_for_answer(last.content[0].text); break
                if last.recipient == 'python':
                    python_calls += 1
                    tool_resp = local_tool.process_sync_plus(last)
                    rt = tool_resp[0].content[0].text
                    if rt.startswith('[ERROR]') or 'Traceback' in rt or 'Error:' in rt: python_errors += 1
                    conversation.messages.extend(tool_resp)

            # Follow-up when no answer (same as v27)
            if final_answer is None and not stop_event.is_set() and time.time() < deadline:
                followup_msg = Message.from_role_and_content(Role.USER, FOLLOWUP_PROMPT)
                conversation.messages.append(followup_msg)
                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
                if max_tokens >= self.cfg.buffer_tokens:
                    stream = self.client.completions.create(
                        model=self.cfg.served_model_name, temperature=0.0,
                        max_tokens=min(max_tokens, 512),
                        prompt=prompt_ids, seed=attempt_seed, stream=True,
                        extra_body={'stop_token_ids': self.stop_token_ids, 'return_token_ids': True}
                    )
                    try:
                        text_chunks = []
                        for chunk in stream:
                            if stop_event.is_set() or time.time() > deadline: break
                            nx = chunk.choices[0].text
                            if nx: text_chunks.append(nx)
                            if '}' in (nx or ''):
                                st = ''.join(text_chunks[-16:])
                                a = self._scan_for_answer(st)
                                if a is not None: final_answer = a; break
                    finally: stream.close()

        except Exception as exc:
            python_errors += 1
        finally:
            if sandbox: sandbox.reset(); self.sandbox_pool.put(sandbox)
        return {'Attempt': attempt_index+1, 'Response Length': total_tokens, 'Python Calls': python_calls,
                'Python Errors': python_errors, 'Entropy': self._compute_mean_entropy(logprobs_buffer), 'Answer': final_answer}

    def _select_answer(self, results):
        aw, av = defaultdict(float), defaultdict(int)
        for r in results:
            a, e = r['Answer'], r['Entropy']
            if a is not None:
                w = 1.0 / max(e, 1e-9)
                aw[a] += w; av[a] += 1
        scored = sorted([{'answer': a, 'votes': av[a], 'score': aw[a]} for a in aw], key=lambda x: x['score'], reverse=True)
        df = pd.DataFrame([(s['answer'], s['votes'], round(s['score'],3)) for s in scored], columns=['Answer','Votes','Score'])
        display(df)
        if not scored: print('\nFinal Answer: 0\n'); return 0
        print(f'\nFinal Answer: {scored[0]["answer"]}\n')
        return scored[0]['answer']

    def _build_strategy_prefs(self, problem):
        """COMPONENT D: Domain-aware preference prompt selection.

        Attempt layout for 8 attempts:
          [0] standard (generic PREF_GENERAL)
          [1] standard (generic PREF_GENERAL)
          [2] domain-specific pref A (top detected domain)
          [3] domain-specific pref B (second domain if score >= 1, else top domain again)
          [4] code_first
          [5] code_first
          [6] small_cases
          [7] small_cases

        This replaces 2 of the 4 generic standard attempts with domain-tuned prompts.
        The code_first and small_cases counts are unchanged from v27.
        """
        ranked = detect_domain(problem)
        print(f'Domain detection: {ranked[:3]}')

        top_domain = ranked[0][0] if ranked[0][1] >= 1 else 'general'
        second_domain = ranked[1][0] if len(ranked) > 1 and ranked[1][1] >= 1 else top_domain

        pref_a = DOMAIN_TO_PREF.get(top_domain, PREF_GENERAL)
        pref_b = DOMAIN_TO_PREF.get(second_domain, PREF_GENERAL)

        print(f'Pref A: {top_domain}, Pref B: {second_domain}')

        return [
            self.cfg.preference_prompt,  # 0: standard
            self.cfg.preference_prompt,  # 1: standard
            pref_a,                      # 2: domain-specific A
            pref_b,                      # 3: domain-specific B
            PREF_CODE_FIRST,             # 4: code_first
            PREF_CODE_FIRST,             # 5: code_first
            PREF_SMALL_CASES,            # 6: small_cases
            PREF_SMALL_CASES,            # 7: small_cases
        ]

    def solve_problem(self, problem):
        print(f'\nProblem: {problem}\n')

        # COMPONENT D: detect domain and build strategy prefs
        strategy_prefs = self._build_strategy_prefs(problem)
        user_inputs = [f'{problem}\n\n{p}' for p in strategy_prefs[:self.cfg.attempts]]

        elapsed = time.time() - self.notebook_start_time
        left = self.cfg.notebook_limit - elapsed
        reserved = max(0, self.problems_remaining - 1) * self.cfg.base_problem_timeout
        budget = min(max(left - reserved, self.cfg.base_problem_timeout), self.cfg.high_problem_timeout)
        deadline = time.time() + budget
        print(f'Budget: {budget:.0f}s | Problems left: {self.problems_remaining}\n')

        tasks = [(self.cfg.system_prompt, i) for i in range(self.cfg.attempts)]
        detailed, valid = [], []
        stop = threading.Event()
        ex = ThreadPoolExecutor(max_workers=self.cfg.workers)
        try:
            futs = [ex.submit(self._process_attempt, user_inputs[ai], sp, ai, stop, deadline) for sp, ai in tasks]
            for f in as_completed(futs):
                try:
                    r = f.result(); detailed.append(r)
                    if r['Answer'] is not None: valid.append(r['Answer'])
                    c = Counter(valid).most_common(1)
                    if c and c[0][1] >= self.cfg.early_stop:
                        stop.set()
                        for ff in futs: ff.cancel()
                        break
                except Exception as e: print(f'Future failed: {e}')
        finally:
            stop.set(); ex.shutdown(wait=True, cancel_futures=True)
            self.problems_remaining = max(0, self.problems_remaining - 1)
        if detailed:
            df = pd.DataFrame(detailed)
            df['Entropy'] = df['Entropy'].round(3)
            df['Answer'] = df['Answer'].astype('Int64')
            display(df)
        if not valid: print('\nResult: 0\n'); return 0
        return self._select_answer(detailed)

    def __del__(self):
        if hasattr(self, 'server_process'): self.server_process.terminate(); self.server_process.wait()
        if hasattr(self, 'log_file'): self.log_file.close()
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try: self.sandbox_pool.get_nowait().close()
                except: pass

In [ ]:
solver = AIMO3Solver(CFG)

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    id_value = id_.item(0)
    question_text = question.item(0)
    gc.disable()
    final_answer = solver.solve_problem(question_text)
    gc.enable()
    gc.collect()
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    candidates = (
        glob.glob('/kaggle/input/competitions/*/test.csv') +
        glob.glob('/kaggle/input/*/test.csv') +
        glob.glob('/kaggle/input/*/*/test.csv')
    )
    test_path = candidates[0] if candidates else '/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv'
    print(f'Test: {test_path}')
    inference_server.run_local_gateway((test_path,))